### Yellow Cab Chaos - Data Cleaning & Preprocessing

Can I actually see where drivers are wasting time in NYC?

In [1]:
import os
import pandas as pd
import glob
import pyarrow.parquet as pq
from datetime import datetime

# List all raw parquet files
raw_files = sorted(glob.glob("../data/raw/yellow_tripdata_*.parquet"))
print(f"Found {len(raw_files)} parquet files")
print(f"Using latest {min(12, len(raw_files))} months for processing")
raw_files = raw_files[-12:]  # Use last 12 months

Found 11 parquet files
Using latest 11 months for processing


In [2]:
# Load all selected files efficiently using PyArrow
print("Loading parquet files...")

columns_to_read = [
    'tpep_pickup_datetime', 'tpep_dropoff_datetime',
    'PULocationID', 'DOLocationID',
    'passenger_count', 'trip_distance', 'total_amount'
]

dataset = pq.ParquetDataset(raw_files)
table = dataset.read(columns=columns_to_read)
df = table.to_pandas()

print(f"Loaded raw data shape: {df.shape}")

Loading parquet files...
Loaded raw data shape: (44417596, 7)


* Data Cleaning

In [3]:
# Dropping rows with missing key values
df = df.dropna(subset=['PULocationID', 'tpep_pickup_datetime'])

In [4]:
# Removing invalid trips
df = df[df['trip_distance'] > 0]
df = df[df['total_amount'] > 0]
df = df[df['passenger_count'] >= 1]

In [5]:
# Filter valid zones (1-265)
df = df[(df['PULocationID'] >= 1) & (df['PULocationID'] <= 265)]

In [6]:
print(f"After cleaning: {df.shape[0]:,} rows")

After cleaning: 32,606,684 rows


In [8]:
# Aggregate to hourly demand per pickup zone
print("Aggregating to hourly demand...")

hourly_demand = df.groupby(['pickup_date', 'pickup_hour', 'PULocationID']).agg(
    trip_count=('PULocationID', 'count'),
    avg_passengers=('passenger_count', 'mean'),
    avg_distance=('trip_distance', 'mean'),
    avg_fare=('total_amount', 'mean')
).reset_index()

hourly_demand.rename(columns={'PULocationID': 'zone_id'}, inplace=True)

print(f"Aggregated shape: {hourly_demand.shape}")

Aggregating to hourly demand...
Aggregated shape: (893448, 7)


In [9]:
# Save processed data
output_path = "../data/processed/hourly_demand.parquet"
hourly_demand.to_parquet(output_path, compression='snappy', index=False)
print(f"✅ Saved processed data to {output_path}")
print(f"Final dataset size: {len(hourly_demand):,} rows")

✅ Saved processed data to ../data/processed/hourly_demand.parquet
Final dataset size: 893,448 rows


In [10]:
# Quick summary
print("\n=== Summary ===")
print(f"Date range: {hourly_demand['pickup_date'].min()} to {hourly_demand['pickup_date'].max()}")
print(f"Total records: {len(hourly_demand):,}")
print(f"Unique zones: {hourly_demand['zone_id'].nunique()}")
print(f"Total trips: {hourly_demand['trip_count'].sum():,}")

# Load zone names for preview
zones = pd.read_csv("../data/external/taxi_zone_lookup.csv")
sample = hourly_demand.head(10).merge(
    zones[['LocationID', 'Borough', 'Zone']], 
    left_on='zone_id', 
    right_on='LocationID'
)
print("\nSample data with zone names:")
print(sample[['pickup_date', 'pickup_hour', 'Borough', 'Zone', 'trip_count']])


=== Summary ===
Date range: 2007-12-05 to 2025-11-30
Total records: 893,448
Unique zones: 262
Total trips: 32,606,684

Sample data with zone names:
  pickup_date  pickup_hour    Borough                       Zone  trip_count
0  2007-12-05           18  Manhattan        Lincoln Square East           1
1  2008-12-31           23     Queens                JFK Airport           1
2  2009-01-01            0     Queens          LaGuardia Airport           1
3  2009-01-01            0  Manhattan  Times Sq/Theatre District           1
4  2009-01-01            0  Manhattan      Upper West Side South           1
5  2009-01-01            8     Queens          LaGuardia Airport           1
6  2009-01-01           12  Manhattan               Central Park           1
7  2009-01-01           14     Queens          LaGuardia Airport           1
8  2024-12-31           20  Manhattan               Clinton East           1
9  2024-12-31           20  Manhattan  West Chelsea/Hudson Yards           1
